### Notebook 范围

### 目的
用清楚的日历月份窗口重新计算 EP100 分波诊断，并区分两类问题：一类是保留的 raw EPFlux vs O3 RMSE，另一类是更适合跨 case 比较的 EPFlux signed error vs O3 signed error。

### 科学问题
W1+W2 是否主导 all-wave EP100？相对于同年 BWCN reference 的 EP100 偏差，是否能解释相对于同年 BWCN reference 的 O3 偏差？这种关系是否依赖 Jan、Feb、Mar、Apr、May 的具体月份？

### 方法
EP100 定义为 `mean(-ep2)`，100 hPa，40-80N，正值表示更强 upward wave activity，不是 divergence。source window 只使用 `Jan/Feb/Mar/Apr/May`；不再使用 `primary` 或 `lead0_30`。O3 signed error 定义为 member O3 mean minus BWCN same-year O3 mean，目标窗口为该 case 初始化日起到 May 30。

### 预期输出
`outputs/figures/03_ep100/` 下输出月窗口热图、按相同初始化月份分组的 raw scatter 和 signed-error scatter；`outputs/tables/03_ep100/` 下输出 member metric 和 correlation 表。

### 解读
raw EPFlux vs RMSE 只作为延续 0008-01 思路的诊断；跨 case 更应该看 EPFlux error vs O3 error，因为 signed error 保留了偏差方向。

### 注意
非一月初始化 case 没有初始化前月份的 hindcast 数据，因此对应 Jan/Feb 等窗口会是 NA。BWCN EP100 reference 使用 longrun omega-corrected 产品，而 hindcast EP100 使用当前 cleaned hindcast no-omega-consistent 产品，所以 EP100 error 是近似 pathway error。


### 导入与公共路径

### 目的
加载公共工具函数、数据路径和绘图库。

### 科学问题
保证 EP100、O3、reference、日期窗口和保存路径与 Extention_analysis 其他 notebook 一致。

### 方法
使用 `hindcast_extension_utils.py` 中的 `load_epflux`、`compute_all_ep100`、`load_bwcn_ep100_reference`、`load_hindcast_o3`、`load_bwcn_reference_o3` 和日期窗口工具。

### 预期输出
打印工作目录和 Hindcast 数据目录。

### 解读
路径正确说明后续代码只读 cleaned hindcast 和 BWCN reference 数据。

### 注意
所有输出只写入 `code_cleaned/Hindcast_experiment/Extention_analysis/outputs`。


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-code-cleaned")

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 定义月窗口、signed error 和绘图辅助函数

### 目的
把 03 的时间窗口限定为清楚的日历月份，并定义 member-level O3 signed error。

### 科学问题
相对于 reference 的 EPFlux 偏差是否对应相对于 reference 的 O3 偏差？这个问题需要 signed error，而不是 RMSE。

### 方法
`MONTH_KEYS = Jan-Feb-Mar-Apr-May`。对于每个 case，只计算初始化月之后可用的月份。O3 mean error 使用 case 初始化日到 May 30 的平均 O3：`member mean O3 - BWCN reference mean O3`。同时保留 `O3_RMSE_target`，用于 raw EPFlux vs RMSE 延续图。

### 预期输出
本代码块只定义函数，不直接输出图表。

### 解读
`O3_mean_error_target < 0` 表示 hindcast 成员比 BWCN reference 更低臭氧；`EP100_wave1_plus_wave2_error > 0` 表示 W1+W2 upward wave activity 强于同年 reference。

### 注意
O3 reference 和 hindcast 用 day-of-year 对齐后取共同日期，避免不同初始化长度导致平均窗口不一致。


In [ ]:
VERSION_TAG = "v6"
MONTH_KEYS = ["Jan", "Feb", "Mar", "Apr", "May"]
MONTH_LABELS_SIMPLE = MONTH_KEYS
fig_dir = figure_dir("03_ep100")
tab_dir = table_dir("03_ep100")


def monthly_windows_for_case(case):
    windows = {}
    for month in MONTH_KEYS:
        if case_month_window_available(case, month):
            windows[month] = MONTH_WINDOWS[month]
    return windows


def case_plot_label(case):
    meta = parse_case_name(case)
    yy = str(meta["year"])[-2:]
    mm = str(meta["init_month"]).zfill(2)
    family = "H-Clim-3D" if meta["config"] == "NOCOUPL" else "H-INT-3D"
    suffix = ""
    if meta["perturbation"] == "v2_large_temperature":
        suffix = "-LP"
    elif meta["perturbation"] == "v3_humidity":
        suffix = "-MP"
    return f"{family}-{yy}-{mm}{suffix}"


def _case_member_ids(da):
    if da is None or "member" not in da.dims:
        return []
    if "member_short" in da.coords:
        return [str(v) for v in da["member_short"].values]
    return [member_short_id(v) for v in da["member"].values]


def compute_o3_signed_errors(case):
    meta = parse_case_name(case)
    o3, o3_date = load_hindcast_o3(case)
    ref, ref_date = load_bwcn_reference_o3(meta["year"])
    if o3 is None or ref is None:
        return pd.DataFrame(columns=["member", "O3_RMSE_target", "O3_mean_error_target", "O3_MA_mean_error", "O3_MA_min_error"])

    def _member_mean_error(window):
        hmask = date_mask_for_case_window(case, o3_date, window[0], window[1])
        rmask = date_mask_for_case_window(f"{meta['year']}-01", ref_date, window[0], window[1])
        if hmask.sum() == 0 or rmask.sum() == 0:
            return None
        hdoy = case_time_doy(case, o3_date)[hmask]
        rdoy = case_time_doy(f"{meta['year']}-01", ref_date)[rmask]
        common = np.intersect1d(hdoy, rdoy)
        if len(common) == 0:
            return None
        hidx = np.where(hmask)[0][np.isin(hdoy, common)]
        ridx = np.where(rmask)[0][np.isin(rdoy, common)]
        h = o3.isel(lead_time=hidx)
        r = ref.isel(lead_time=ridx)
        # Common DOY is sorted by intersect1d; both hidx/ridx are subset in source order, matching the same DOYs.
        hmean = h.mean("lead_time", skipna=True)
        rmean = float(r.mean("lead_time", skipna=True).values)
        return hmean - rmean

    target = target_window_for_case(case)
    target_error = _member_mean_error(target)
    ma_error = _member_mean_error(((3, 1), (4, 30)))

    rmse = compute_o3_rmse(o3, ref, target[0], target[1]).rename(columns={"O3_RMSE": "O3_RMSE_target"})
    out = rmse.copy()
    if target_error is not None:
        out = out.merge(pd.DataFrame({"member": _case_member_ids(target_error), "O3_mean_error_target": target_error.values.astype(float)}), on="member", how="outer")
    else:
        out["O3_mean_error_target"] = np.nan
    if ma_error is not None:
        out = out.merge(pd.DataFrame({"member": _case_member_ids(ma_error), "O3_MA_mean_error": ma_error.values.astype(float)}), on="member", how="outer")
    else:
        out["O3_MA_mean_error"] = np.nan

    # March-April minimum signed error is useful as a more event-severity-like diagnostic.
    hmask = date_mask_for_case_window(case, o3_date, (3, 1), (4, 30))
    rmask = date_mask_for_case_window(f"{meta['year']}-01", ref_date, (3, 1), (4, 30))
    if hmask.sum() and rmask.sum():
        hmin = o3.isel(lead_time=hmask).min("lead_time", skipna=True)
        rmin = float(ref.isel(lead_time=rmask).min("lead_time", skipna=True).values)
        out = out.merge(pd.DataFrame({"member": _case_member_ids(hmin), "O3_MA_min_error": hmin.values.astype(float) - rmin}), on="member", how="outer")
    else:
        out["O3_MA_min_error"] = np.nan
    return out


def add_ep100_reference_errors(ep, year, window):
    if ep.empty:
        return ep
    out = ep.copy()
    for wave in WAVES:
        col = f"EP100_{wave}"
        ref_val = load_bwcn_ep100_reference(year, wave, window)
        out[f"{col}_reference"] = ref_val
        out[f"{col}_error"] = out[col] - ref_val if col in out else np.nan
    ref_w1 = out["EP100_wave1_reference"] if "EP100_wave1_reference" in out else np.nan
    ref_w2 = out["EP100_wave2_reference"] if "EP100_wave2_reference" in out else np.nan
    out["EP100_wave1_plus_wave2_reference"] = ref_w1 + ref_w2
    out["EP100_wave1_plus_wave2_error"] = out["EP100_wave1_plus_wave2"] - out["EP100_wave1_plus_wave2_reference"]
    return out


def finite_line_label(x, y):
    c = finite_corr(x, y)
    if c["n"] < 3 or not np.isfinite(c["R"]):
        return "R=NA, p=NA, n=%d" % c["n"], c
    return f"R={c['R']:.2f}, p={c['p']:.2e}, n={c['n']}", c


### 计算每个 case-member-month 的 EP100 与 O3 指标

### 目的
为每个可 source-diagnose 的 hindcast case、每个可用日历月份、每个成员生成统一指标表。

### 科学问题
在相同日历月份内，member EP100 的绝对值和相对于 BWCN 的偏差，分别如何对应 O3 RMSE 和 O3 signed error？

### 方法
对每个 case 的 Jan/Feb/Mar/Apr/May 可用月份计算 all waves、wave1、wave2、rest、W1+W2 的 EP100。再减去同年 BWCN reference EP100 得到 signed EP100 error。O3 指标包括 init-May30 RMSE、init-May30 mean error、Mar-Apr mean error、Mar-Apr minimum error。

### 预期输出
`03_EP100_monthly_case_member_metrics_v4.csv`，并保留 root-level `03_EP100_case_member_metrics.csv` 供后续 synthesis 读取。

### 解读
该表是所有 03 图的源数据；检查 `source_month` 可以确认横轴只剩 Jan-Feb-Mar-Apr-May。

### 注意
如果某个 case 初始化较晚，初始化前月份不会出现在该 case 的表中。


In [ ]:
inv = discover_hindcast_cases()
all_metrics = []
for case in inv.loc[inv["can_source_diagnose"], "case"]:
    meta = parse_case_name(case)
    o3_errors = compute_o3_signed_errors(case)
    for month, window in monthly_windows_for_case(case).items():
        ep = compute_all_ep100(case, window)
        if ep.empty:
            log_message(f"{case}: empty EP100 for {month} in 03 monthly metrics")
            continue
        ep = add_ep100_reference_errors(ep, meta["year"], window)
        tbl = ep.merge(o3_errors, on="member", how="outer")
        tbl["case"] = case
        tbl["plot_label"] = case_plot_label(case)
        tbl["year"] = meta["year"]
        tbl["init_month"] = meta["init_month"]
        tbl["config"] = meta["config"]
        tbl["perturbation"] = meta["perturbation"]
        tbl["source_month"] = month
        tbl["source_window"] = window_to_label(window)
        tbl["target_window"] = window_to_label(target_window_for_case(case))
        tbl["EP100_definition"] = "mean(-ep2), 100 hPa, 40-80N; positive=stronger upward wave activity; not divergence"
        tbl["O3_error_definition"] = "member O3 mean minus BWCN same-year O3 mean over target window"
        all_metrics.append(tbl)

metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
metrics["source_month"] = pd.Categorical(metrics["source_month"], categories=MONTH_KEYS, ordered=True) if not metrics.empty else pd.Series(dtype="category")
metrics_csv = tab_dir / f"03_EP100_monthly_case_member_metrics_{VERSION_TAG}.csv"
metrics.to_csv(metrics_csv, index=False)
# Historical root filename; now monthly, no primary/lead windows.
metrics.to_csv(TAB_DIR / "03_EP100_case_member_metrics.csv", index=False)
print(metrics[["case", "plot_label", "member", "source_month", "EP100_wave1_plus_wave2", "EP100_wave1_plus_wave2_error", "O3_RMSE_target", "O3_mean_error_target"]].head(20).to_string(index=False))
print("rows", len(metrics), "cases", metrics["case"].nunique() if not metrics.empty else 0)


### 绘制月窗口相关热图

### 目的
把原先混合 `primary/lead0_30/Jan/Feb` 的 window sensitivity 图改成纯月份横轴：Jan、Feb、Mar、Apr、May。

### 科学问题
EP100 W1+W2 的 reference-relative error 是否与 O3 signed error 相关？all-wave EP100 是否主要由 W1+W2 解释？这些关系是否随月份改变？

### 方法
每个 case、每个月份，在成员维度上计算 Pearson R。主热图使用 `R(EP100_W1+W2_error, O3_mean_error_target)`；dominance 热图使用 `R(EP100_all_waves, EP100_W1+W2)`，保留绝对值用于判断 all-wave 是否被 W1+W2 主导。

### 预期输出
`03_EP100_W1W2_error_vs_O3_error_monthly_sensitivity_v4.png/pdf` 和 `03_all_waves_vs_W1W2_monthly_sensitivity_v4.png/pdf`；支撑表为 `03_EP100_monthly_correlations_v6.csv`。

### 解读
第一张图中负相关表示更强 upward W1+W2 wave activity error 对应更低 O3 error；第二张图高正相关表示 all-wave EP100 主要随 W1+W2 变化。

### 注意
May 的 EPFlux 与 Mar-Apr O3 可能是同期或偏晚关系，应主要作为敏感性检查，不应解释为 precursor。


In [ ]:
corr_rows = []
if not metrics.empty:
    for (case, month), sub in metrics.groupby(["case", "source_month"], observed=True):
        relationships = {
            "EP100_W1W2_error vs O3_mean_error_target": ("EP100_wave1_plus_wave2_error", "O3_mean_error_target"),
            "EP100_W1W2_raw vs O3_RMSE_target": ("EP100_wave1_plus_wave2", "O3_RMSE_target"),
            "all_waves_raw vs W1W2_raw": ("EP100_all_waves", "EP100_wave1_plus_wave2"),
            "all_waves_error vs W1W2_error": ("EP100_all_waves_error", "EP100_wave1_plus_wave2_error"),
        }
        for rel, (xcol, ycol) in relationships.items():
            c = finite_corr(sub[xcol], sub[ycol]) if xcol in sub and ycol in sub else {"R": np.nan, "p": np.nan, "n": 0}
            corr_rows.append({"case": case, "source_month": month, "relationship": rel, **c})

corr = pd.DataFrame(corr_rows)
corr_csv = tab_dir / f"03_EP100_monthly_correlations_{VERSION_TAG}.csv"
corr.to_csv(corr_csv, index=False)
# Root copies for downstream synthesis; now explicitly monthly and signed-error aware.
corr.to_csv(TAB_DIR / "03_EP100_O3_U_FWD_correlations_all_cases.csv", index=False)
corr[corr["relationship"].isin(["all_waves_raw vs W1W2_raw", "all_waves_error vs W1W2_error"])].to_csv(TAB_DIR / "03_wave_component_relevance_all_cases.csv", index=False)


def plot_corr_heatmap(relationship, name, title, interpretation):
    sub = corr[corr["relationship"] == relationship]
    mat = sub.pivot(index="case", columns="source_month", values="R") if not sub.empty else pd.DataFrame()
    fig, ax = plt.subplots(figsize=(9.5, max(4.5, 0.32 * len(mat) + 2)), constrained_layout=True)
    if mat.empty:
        ax.axis("off")
        ax.text(0.5, 0.5, "No monthly EP100 correlations", ha="center")
    else:
        cols = [m for m in MONTH_KEYS if m in mat.columns]
        mat = mat[cols]
        im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(mat.shape[1]), mat.columns)
        ax.set_yticks(range(mat.shape[0]), mat.index)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat.iloc[i, j]
                ax.text(j, i, "NA" if not np.isfinite(val) else f"{val:.2f}", ha="center", va="center", fontsize=8)
        ax.set_title(title + "\nEP100=mean(-ep2), 100 hPa, 40-80N; monthly windows only")
        fig.colorbar(im, ax=ax, label="Pearson R")
    savefig(
        fig,
        name,
        fig_dir=fig_dir,
        notebook="03_EP100_wave_component_multicase.ipynb",
        scientific_question="EP100-O3 或 all-wave-W1W2 关系是否依赖日历月份？",
        variables_windows="Monthly source windows Jan-Feb-Mar-Apr-May; EP100 mean(-ep2), 100 hPa, 40-80N; signed errors are relative to BWCN same-year reference.",
        interpretation=interpretation,
        caveat="非一月初始化 case 缺少初始化前月份；May 更适合敏感性检查而不是 precursor 解释。",
        csv_table=corr_csv,
    )
    plt.close(fig)

plot_corr_heatmap(
    "EP100_W1W2_error vs O3_mean_error_target",
    f"03_EP100_W1W2_error_vs_O3_error_monthly_sensitivity_{VERSION_TAG}",
    "R(EP100 W1+W2 error, signed O3 mean error) by calendar month",
    "这是主 signed-error 图；它比 RMSE 更能显示偏差方向。",
)
plot_corr_heatmap(
    "all_waves_raw vs W1W2_raw",
    f"03_all_waves_vs_W1W2_monthly_sensitivity_{VERSION_TAG}",
    "R(EP100 all waves, EP100 W1+W2) by calendar month",
    "高相关说明 all-wave EP100 主要由 W1+W2 控制。",
)
print(corr.head(20).to_string(index=False))


### 保留 raw EPFlux vs O3 RMSE，但按相同初始化月份分组

### 目的
保留原来的 raw EPFlux vs O3 RMSE 诊断，但不再把不同初始化日期混在同一张 selected-case 图里。

### 科学问题
在相同初始化月份内部，raw EP100 W1+W2 的成员差异是否对应 O3 RMSE？这种关系在 Jan、Feb、Mar、Apr、May 哪个月份最明显？

### 方法
按 `init_month` 分组，每个图包含同一初始化月份的所有 case。每个 panel 是一个 source month；点的颜色表示 case。x 为 raw `EP100_wave1_plus_wave2`，y 为 `O3_RMSE_target`。标题明确写出该 panel 的月份、R、p、n；p 值使用科学计数法，图例放在图片下方。

### 预期输出
`03_RAW_EP100_W1W2_vs_O3_RMSE_initXX_monthly_v4.png/pdf`。

### 解读
这个图只是保留 0008-01 思路的 raw 版本；如果相关性只在极端事件中出现，说明 raw EPFlux 绝对值不适合跨事件泛化。

### 注意
RMSE 没有正负方向，因此不能判断 O3 是偏高还是偏低；跨 case 机制解释应优先看下一块 signed-error 图。


In [ ]:
def _scatter_by_init_month(xcol, ycol, prefix, xlab, ylab, question, interpretation):
    if metrics.empty:
        return []
    outputs = []
    for init_month, init_sub in metrics.groupby("init_month"):
        available_months = [m for m in MONTH_KEYS if m in set(init_sub["source_month"].astype(str))]
        if not available_months:
            continue
        ncols = len(available_months)
        fig, axes = plt.subplots(1, ncols, figsize=(4.9 * ncols, 5.15), squeeze=False)
        case_list = sorted(init_sub["case"].unique())
        cmap = plt.get_cmap("tab20")
        colors = {case: cmap(i % 20) for i, case in enumerate(case_list)}
        case_labels = {case: case_plot_label(case) for case in case_list}
        legend_handles, legend_labels = [], []
        for ax, month in zip(axes.ravel(), available_months):
            subm = init_sub[init_sub["source_month"].astype(str) == month].copy()
            finite_mask = np.isfinite(subm[xcol].astype(float)) & np.isfinite(subm[ycol].astype(float))
            if not finite_mask.any():
                ax.text(0.5, 0.5, f"No finite {ylab} data", transform=ax.transAxes, ha="center", va="center", fontsize=10, color="0.35")
            for case, csub in subm.groupby("case"):
                valid = csub[np.isfinite(csub[xcol].astype(float)) & np.isfinite(csub[ycol].astype(float))]
                if valid.empty:
                    continue
                label_text = case_labels.get(case, case)
                artist = ax.scatter(valid[xcol], valid[ycol], s=42, alpha=0.82, edgecolor="black", linewidth=0.4, color=colors[case], label=label_text)
                if label_text not in legend_labels:
                    legend_handles.append(artist)
                    legend_labels.append(label_text)
            valid_sub = subm.loc[finite_mask]
            label, c = finite_line_label(valid_sub[xcol], valid_sub[ycol])
            ax.set_title(f"{month}: {label}", fontsize=10)
            ax.set_xlabel(xlab, fontsize=9)
            ax.set_ylabel(ylab, fontsize=9)
            ax.grid(True, alpha=0.25)
        if legend_handles:
            fig.legend(legend_handles, legend_labels, loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=min(6, len(legend_labels)), fontsize=8, frameon=True)
        fig.suptitle(f"Initialization month {init_month}: {ylab} vs {xlab}", fontsize=13, fontweight="bold", y=0.97)
        fig.subplots_adjust(left=0.07, right=0.99, top=0.82, bottom=0.27, wspace=0.34)
        csv = metrics_csv
        name = f"{prefix}_init{init_month}_monthly_{VERSION_TAG}"
        savefig(
            fig,
            name,
            fig_dir=fig_dir,
            notebook="03_EP100_wave_component_multicase.ipynb",
            scientific_question=question,
            variables_windows="Same initialization month only; monthly EP100 source windows Jan-Feb-Mar-Apr-May; EP100=mean(-ep2), 100 hPa, 40-80N.",
            interpretation=interpretation,
            caveat="Pooling is only within the same initialization month; RMSE plots do not retain signed O3 bias direction.",
            csv_table=csv,
        )
        plt.close(fig)
        outputs.append(name)
    return outputs

raw_outputs = _scatter_by_init_month(
    xcol="EP100_wave1_plus_wave2",
    ycol="O3_RMSE_target",
    prefix="03_RAW_EP100_W1W2_vs_O3_RMSE",
    xlab="Raw EP100 W1+W2\nmean(-ep2), 100 hPa, 40-80N",
    ylab="O3 RMSE, init-May30 (DU)",
    question="同一初始化月份内，raw EP100 W1+W2 是否对应 O3 RMSE？",
    interpretation="这是保留的 raw/RMSE 版本；适合看 0008-01 式极端情形，但不适合作为跨事件主证据。",
)
print("raw scatter outputs", raw_outputs)


### 绘制 EPFlux signed error vs O3 signed error

### 目的
新增更适合跨事件比较的主图：平均 EP100 error vs 平均 O3 error，保留正负方向。

### 科学问题
相对于同年 BWCN reference，W1+W2 upward wave activity 偏强或偏弱，是否对应 O3 平均值偏高或偏低？这种关系在相同初始化月份内是否稳定？

### 方法
按 `init_month` 分组，每个 panel 是一个 source month。x 为 `EP100_wave1_plus_wave2_error = member EP100 - BWCN reference EP100`；y 为 `O3_mean_error_target = member O3 mean - BWCN reference O3 mean`，目标窗口为该 case 初始化日到 May 30。图例放在图片下方，避免遮盖点云；case 标签统一使用 `H-INT-3D-YY-MM-LP/MP` 或 `H-Clim-3D-YY-MM`。

### 预期输出
`03_EP100_W1W2_error_vs_O3_mean_error_initXX_monthly_v4.png/pdf`。

### 解读
x>0 表示成员的 upward W1+W2 wave activity 强于 reference；y<0 表示成员 O3 低于 reference。若出现稳定负相关，说明更强的 W1+W2 wave activity error 可能对应更低的 O3 pathway error。

### 注意
这仍是成员间相关，不是因果证明；月份越晚越可能混入臭氧反馈后的结果，不能简单解释为 precursor。


In [ ]:
signed_outputs = _scatter_by_init_month(
    xcol="EP100_wave1_plus_wave2_error",
    ycol="O3_mean_error_target",
    prefix="03_EP100_W1W2_error_vs_O3_mean_error",
    xlab="EP100 W1+W2 error\nmember - BWCN reference",
    ylab="O3 mean error, init-May30 (DU)\nmember - BWCN reference",
    question="同一初始化月份内，EP100 W1+W2 signed error 是否对应 O3 signed error？",
    interpretation="signed-error 图保留偏差方向，是跨事件比较的主图。",
)
print("signed scatter outputs", signed_outputs)
write_figure_guide()
